# Medical VQA — Image-Only CNN Baseline
**Member 2 | Google Colab Version**

This notebook trains an image-only CNN baseline for binary yes/no VQA on medical X-ray images.

Before running:
- Set runtime to **GPU**: `Runtime → Change runtime type → T4 GPU`
- Run all cells top to bottom

## 1. Install Dependencies

In [1]:
!pip install -q datasets transformers torch torchvision scikit-learn pillow

## 2. Mount Google Drive (for saving model)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# All model checkpoints will be saved here
SAVE_DIR = '/content/drive/MyDrive/Medical_VQA'

import os
os.makedirs(SAVE_DIR, exist_ok=True)
print('Save directory ready:', SAVE_DIR)

Mounted at /content/drive
Save directory ready: /content/drive/MyDrive/Medical_VQA


## 3. Imports

In [3]:
import io
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


## 4. Utility Functions (from src/utils.py)

In [4]:
# --- Image helpers ---
def open_image_from_dataset_value(image_value):
    if isinstance(image_value, Image.Image):
        return image_value
    if isinstance(image_value, dict) and image_value.get('bytes') is not None:
        return Image.open(io.BytesIO(image_value['bytes']))
    if isinstance(image_value, dict) and image_value.get('path') is not None:
        return Image.open(image_value['path'])
    return None

# Standard preprocessing — ResNet/ImageNet compatible
IMAGE_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Lambda(lambda img: img.convert('RGB')),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# --- PyTorch Dataset ---
class MedicalVQABinaryDataset(Dataset):
    def __init__(self, dataframe, image_col, question_col, label_col='label', transform=None):
        self.dataframe   = dataframe.reset_index(drop=True)
        self.image_col   = image_col
        self.question_col = question_col
        self.label_col   = label_col
        self.transform   = transform if transform is not None else IMAGE_TRANSFORM

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        img = open_image_from_dataset_value(row[self.image_col])
        if img is None:
            raise ValueError(f'Could not open image at index {idx}')
        img      = self.transform(img)
        question = str(row[self.question_col])
        label    = torch.tensor(row[self.label_col], dtype=torch.long)
        return {'image': img, 'question': question, 'label': label}

# --- Evaluation helpers ---
def compute_metrics(y_true, y_pred, y_prob=None):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1':       f1_score(y_true, y_pred, zero_division=0),
        'auc-roc':  roc_auc_score(y_true, y_prob) if y_prob is not None else None,
    }

def print_metrics(metrics, model_name='Model'):
    print(f"\n{'='*40}")
    print(f'  {model_name}')
    print(f"{'='*40}")
    print(f"  Accuracy : {metrics['accuracy']:.4f}")
    print(f"  F1 Score : {metrics['f1']:.4f}")
    if metrics.get('auc-roc') is not None:
        print(f"  AUC-ROC  : {metrics['auc-roc']:.4f}")
    print(f"{'='*40}\n")

print('Utilities ready!')

Utilities ready!


## 5. Load Dataset

In [5]:
# Load from HuggingFace
dataset = load_dataset('robailleo/medical-vision-llm-dataset')

# Convert to DataFrames
train_df = dataset['train'].to_pandas()
val_df   = dataset['validation'].to_pandas()

# Filter to binary yes/no only
train_df = train_df[train_df['answer'].str.lower().isin(['yes', 'no'])].reset_index(drop=True)
val_df   = val_df[val_df['answer'].str.lower().isin(['yes', 'no'])].reset_index(drop=True)

# Add numeric label column
train_df['label'] = train_df['answer'].str.lower().map({'yes': 1, 'no': 0})
val_df['label']   = val_df['answer'].str.lower().map({'yes': 1, 'no': 0})

print('Train size:', len(train_df))
print('Val size:  ', len(val_df))
print('Label balance (train):')
print(train_df['label'].value_counts())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004.parquet:   0%|          | 0.00/146M [00:00<?, ?B/s]

data/train-00001-of-00004.parquet:   0%|          | 0.00/149M [00:00<?, ?B/s]

data/train-00002-of-00004.parquet:   0%|          | 0.00/153M [00:00<?, ?B/s]

data/train-00003-of-00004.parquet:   0%|          | 0.00/159M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/155M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3834 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/959 [00:00<?, ? examples/s]

Train size: 750
Val size:   190
Label balance (train):
label
0    379
1    371
Name: count, dtype: int64


## 6. Build DataLoaders

In [6]:
train_dataset = MedicalVQABinaryDataset(train_df, image_col='image', question_col='question', label_col='label')
val_dataset   = MedicalVQABinaryDataset(val_df,   image_col='image', question_col='question', label_col='label')

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print('Train batches:', len(train_loader))
print('Val batches:  ', len(val_loader))

Train batches: 47
Val batches:   12


## 7. CNN Model

In [12]:
import torchvision.models as models

# Load pretrained ResNet-18
model = models.resnet18(weights = "IMAGENET1K_V1")

# Freeze all layers
for param in model.parameters():
  param.requires_grad = False

# Unfreeze last residual block
for param in model.layer4.parameters():
  param.requires_grad = True

# Replace final lauer with binary head
model.fc = nn.Linear(model.fc.in_features, 1)

# Only new fc layer is trainable
model = model.to(device)
print("Model ready on:", device)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

Model ready on: cuda
Trainable params: 8394241


## 8. Training Loop

In [8]:
def train(model, train_loader, val_loader, epochs=20, patience=5):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6
    )

    best_val_acc     = 0.0
    patience_counter = 0
    save_path        = os.path.join(SAVE_DIR, 'cnn_best.pth')

    for epoch in range(epochs):
        # --- Training ---
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0

        for batch in train_loader:
            images = batch['image'].to(device)
            labels = batch['label'].float().to(device)

            optimizer.zero_grad()
            outputs = model(images).squeeze(1)
            loss    = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss    += loss.item()
            preds          = (torch.sigmoid(outputs) >= 0.5).long()
            train_correct += (preds == labels.long()).sum().item()
            train_total   += labels.size(0)

        # --- Validation ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for batch in val_loader:
                images = batch['image'].to(device)
                labels = batch['label'].float().to(device)

                outputs = model(images).squeeze(1)
                loss    = criterion(outputs, labels)

                val_loss    += loss.item()
                preds        = (torch.sigmoid(outputs) >= 0.5).long()
                val_correct += (preds == labels.long()).sum().item()
                val_total   += labels.size(0)

        train_acc = 100 * train_correct / train_total
        val_acc   = 100 * val_correct   / val_total
        val_loss  /= len(val_loader)

        scheduler.step(val_loss)
        lr = optimizer.param_groups[0]['lr']

        print(f'Epoch [{epoch+1:02d}/{epochs}] '
              f'Train Acc: {train_acc:.2f}% | '
              f'Val Acc: {val_acc:.2f}% | '
              f'Val Loss: {val_loss:.4f} | LR: {lr:.6f}')

        if val_acc > best_val_acc:
            best_val_acc     = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
            print(f'  ✓ Best model saved (Val Acc: {val_acc:.2f}%)')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'\nEarly stopping at epoch {epoch+1}.')
                break

    print(f'\nBest Val Accuracy: {best_val_acc:.2f}%')
    return save_path

print('Training function ready!')

Training function ready!


## 9. Train the Model

In [13]:
torch.manual_seed(RANDOM_STATE)
save_path = train(model, train_loader, val_loader, epochs=20, patience=5)

Epoch [01/20] Train Acc: 60.40% | Val Acc: 57.37% | Val Loss: 1.3314 | LR: 0.001000
  ✓ Best model saved (Val Acc: 57.37%)
Epoch [02/20] Train Acc: 66.80% | Val Acc: 61.05% | Val Loss: 0.8252 | LR: 0.001000
  ✓ Best model saved (Val Acc: 61.05%)
Epoch [03/20] Train Acc: 71.33% | Val Acc: 59.47% | Val Loss: 0.6764 | LR: 0.001000
Epoch [04/20] Train Acc: 73.20% | Val Acc: 70.00% | Val Loss: 0.6761 | LR: 0.001000
  ✓ Best model saved (Val Acc: 70.00%)
Epoch [05/20] Train Acc: 77.33% | Val Acc: 67.89% | Val Loss: 0.6567 | LR: 0.001000
Epoch [06/20] Train Acc: 79.33% | Val Acc: 67.89% | Val Loss: 0.6691 | LR: 0.001000
Epoch [07/20] Train Acc: 78.27% | Val Acc: 65.26% | Val Loss: 0.7900 | LR: 0.001000
Epoch [08/20] Train Acc: 80.00% | Val Acc: 69.47% | Val Loss: 0.6314 | LR: 0.001000
Epoch [09/20] Train Acc: 80.67% | Val Acc: 72.63% | Val Loss: 0.6877 | LR: 0.001000
  ✓ Best model saved (Val Acc: 72.63%)
Epoch [10/20] Train Acc: 79.73% | Val Acc: 69.47% | Val Loss: 0.7278 | LR: 0.001000
Epoc

## 10. Evaluate Best Model

In [14]:
# Load best checkpoint
model.load_state_dict(torch.load(save_path, map_location=device))
model.eval()

all_preds, all_probs, all_labels = [], [], []

with torch.no_grad():
    for batch in val_loader:
        images = batch['image'].to(device)
        labels = batch['label']

        outputs = model(images).squeeze(1)
        probs   = torch.sigmoid(outputs).cpu()
        preds   = (probs >= 0.5).long()

        all_preds.extend(preds.tolist())
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.tolist())

metrics_cnn = compute_metrics(all_labels, all_preds, all_probs)
print_metrics(metrics_cnn, model_name='Image-Only CNN Baseline')


  Image-Only CNN Baseline
  Accuracy : 0.7263
  F1 Score : 0.7500
  AUC-ROC  : 0.7740



## 11. Results Summary

In [15]:
summary = pd.DataFrame([{
    'Model':    'Image-Only CNN Baseline',
    'Accuracy': round(metrics_cnn['accuracy'], 4),
    'F1':       round(metrics_cnn['f1'], 4),
    'AUC-ROC':  round(metrics_cnn['auc_roc'], 4),
}])

print(summary.to_string(index=False))

                  Model  Accuracy   F1  AUC-ROC
Image-Only CNN Baseline    0.7263 0.75    0.774
